# Semantic Entropy: Critical Evaluation on Real LLM Outputs

**Sheroz Khan** | iRisk Lab, UIUC | Fall 2026

## Week 2: Real Generations, Harder Questions, and Baseline Comparison

**Paper:** Farquhar, S., Kossen, J., Kuhn, L., and Gal, Y. (2024). "Detecting Hallucinations in Large Language Models Using Semantic Entropy." *Nature*, 630, 625-630. https://doi.org/10.1038/s41586-024-07421-0

---

### Week 1 Recap

Week 1 implemented the full SE pipeline (NLI clustering via DeBERTa, discrete entropy) on simulated generations and confirmed the method works on easy factoid questions. Two key findings:
1. The NLI model cannot recognize numerical equivalence across units ("299,792,458 m/s" vs "3 x 10^8 m/s"), inflating SE on correct answers.
2. For well-known facts, SE is zero regardless of phrasing variation, which is the paper's core claim working correctly.

### Week 2 Goals

1. **Fix the Ollama/Gemma temperature issue** and generate real (non-simulated) LLM outputs.
2. **Add harder questions** where the model is likely to hallucinate, to test whether SE actually flags confabulation.
3. **Add a baseline comparison:** unique string count vs. semantic entropy, to quantify the value of NLI clustering over naive string matching.

## Environment Setup

Same dependencies as Week 1, plus a script to fix Ollama temperature.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import math
import json
import time
import requests
import subprocess
import warnings
warnings.filterwarnings('ignore')

print("Base packages loaded.")

## Load NLI Model (same as Week 1)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

NLI_MODEL_NAME = "microsoft/deberta-large-mnli"

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)
nli_model.eval()

NLI_LABELS = {0: "contradiction", 1: "neutral", 2: "entailment"}
print(f"NLI model loaded: {NLI_MODEL_NAME}")

## Core Functions (from Week 1)

In [ ]:
def check_entailment(premise: str, hypothesis: str) -> str:
    inputs = nli_tokenizer(premise, hypothesis, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        logits = nli_model(**inputs).logits
    return NLI_LABELS[logits.argmax(dim=-1).item()]


def are_semantically_equivalent(s_a: str, s_b: str, context: str = "") -> bool:
    if context:
        s_a_ctx = f"Question: {context} Answer: {s_a}"
        s_b_ctx = f"Question: {context} Answer: {s_b}"
    else:
        s_a_ctx, s_b_ctx = s_a, s_b
    forward = check_entailment(s_a_ctx, s_b_ctx)
    backward = check_entailment(s_b_ctx, s_a_ctx)
    return (forward == "entailment") and (backward == "entailment")


def cluster_by_meaning(generations: list, context: str = "") -> list:
    clusters = []
    for gen in generations:
        assigned = False
        for cluster in clusters:
            if are_semantically_equivalent(gen, cluster[0], context):
                cluster.append(gen)
                assigned = True
                break
        if not assigned:
            clusters.append([gen])
    return clusters


def compute_semantic_entropy(clusters: list, n_total: int) -> float:
    entropy = 0.0
    for cluster in clusters:
        p_k = len(cluster) / n_total
        if p_k > 0:
            entropy -= p_k * math.log(p_k)
    return entropy


print("Core functions loaded.")

## Fixing Ollama Temperature

Ollama with Gemma3:4b ignores the `temperature` parameter passed via the API in some releases. The fix is to create a custom model via a Modelfile that hardcodes sampling parameters at the model level.

Run this **once** in your terminal before running this notebook:

```bash
cat > /tmp/Modelfile << 'EOF'
FROM gemma3:4b
PARAMETER temperature 1.5
PARAMETER top_p 0.95
PARAMETER top_k 50
PARAMETER repeat_penalty 1.2
EOF

ollama create gemma3-diverse -f /tmp/Modelfile
```

Or run the cell below to do it programmatically.

In [ ]:
# Create a custom Ollama model with temperature baked in
MODELFILE_CONTENT = """FROM gemma3:4b
PARAMETER temperature 1.5
PARAMETER top_p 0.95
PARAMETER top_k 50
PARAMETER repeat_penalty 1.2
"""

DIVERSE_MODEL = "gemma3-diverse"

def create_diverse_model():
    """Create the custom temperature-fixed model in Ollama."""
    try:
        # Check if it already exists
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        models = [m["name"] for m in r.json().get("models", [])]
        if any(DIVERSE_MODEL in m for m in models):
            print(f"Model '{DIVERSE_MODEL}' already exists. Skipping creation.")
            return True

        # Create via API
        r = requests.post("http://localhost:11434/api/create", json={
            "name": DIVERSE_MODEL,
            "modelfile": MODELFILE_CONTENT,
        }, timeout=300, stream=True)

        # Stream the response (creation can take a moment)
        for line in r.iter_lines():
            if line:
                status = json.loads(line).get("status", "")
                if status:
                    print(f"  {status}")

        print(f"Model '{DIVERSE_MODEL}' created successfully.")
        return True

    except Exception as e:
        print(f"Failed to create model: {e}")
        print("Create it manually: ollama create gemma3-diverse -f /tmp/Modelfile")
        return False


model_created = create_diverse_model()

## Generation Function

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/generate"

def generate_samples(
    question: str,
    n_samples: int = 10,
    model: str = DIVERSE_MODEL,
    max_tokens: int = 100,
) -> list:
    """
    Generate n_samples from Ollama.
    Temperature is baked into the model via Modelfile, not passed as a parameter.
    """
    system_prompt = "Answer the question concisely. Give only the answer, no explanation or reasoning."
    prompt = f"{system_prompt}\n\nQuestion: {question}\nAnswer:"

    generations = []
    for i in range(n_samples):
        try:
            resp = requests.post(OLLAMA_URL, json={
                "model": model,
                "prompt": prompt,
                "stream": False,
                "options": {"num_predict": max_tokens},
            }, timeout=120)
            resp.raise_for_status()
            text = resp.json().get("response", "").strip()
            # Take first line only
            text = text.split("\n")[0].strip()
            if text:
                generations.append(text)
        except Exception as e:
            print(f"  Sample {i+1} failed: {e}")

    return generations


# === Quick diversity test ===
def test_diversity():
    print("Testing generation diversity...")
    samples = generate_samples("What is the capital of France?", n_samples=5)
    print(f"  Samples: {samples}")
    unique = len(set(samples))
    print(f"  Unique: {unique}/{len(samples)}")
    if unique <= 1:
        print("  WARNING: No diversity. Temperature fix may not have worked.")
        print("  Try: ollama create gemma3-diverse -f /tmp/Modelfile")
        print("  Or switch to llama3.2:3b")
    else:
        print("  Diversity confirmed.")
    return unique > 1

diversity_ok = test_diversity()

In [ ]:
# If diversity test failed, fall back to llama3.2:3b or simulated
FALLBACK_MODEL = "llama3.2:3b"

if not diversity_ok:
    print(f"Trying fallback model: {FALLBACK_MODEL}")
    print("Pull it first if needed: ollama pull llama3.2:3b")
    try:
        test = generate_samples("What is 2+2?", n_samples=3, model=FALLBACK_MODEL)
        if len(set(test)) > 1:
            print(f"Fallback works with diversity. Switching to {FALLBACK_MODEL}.")
            DIVERSE_MODEL = FALLBACK_MODEL
            diversity_ok = True
        else:
            print("Fallback also lacks diversity. Will use simulated generations.")
    except:
        print("Fallback model not available. Will use simulated generations.")

## Question Sets

### Set A: Easy (from Week 1)
Factoid questions where the model should be confident. Expected: low SE.

### Set B: Hard / Hallucination-Prone
Questions where LLMs commonly confabulate: obscure facts, numerical reasoning, trick questions, ambiguous queries. Expected: higher SE if the model is uncertain, or low SE with wrong answer if the model is confidently wrong (which SE would miss).

In [ ]:
QUESTIONS_EASY = [
    {"question": "What is the capital of France?",                    "answer": "Paris"},
    {"question": "Who painted the Mona Lisa?",                        "answer": "Leonardo da Vinci"},
    {"question": "What is the chemical formula for water?",           "answer": "H2O"},
    {"question": "In what year did World War II end?",                "answer": "1945"},
    {"question": "Who wrote Romeo and Juliet?",                       "answer": "William Shakespeare"},
]

QUESTIONS_HARD = [
    # Obscure facts: model likely to guess/confabulate
    {"question": "Who was the second person to walk on the Moon?",
     "answer": "Buzz Aldrin"},
    {"question": "What is the capital of Myanmar?",
     "answer": "Naypyidaw"},
    {"question": "In what year was the Treaty of Tordesillas signed?",
     "answer": "1494"},
    {"question": "What is the atomic number of Selenium?",
     "answer": "34"},
    # Trick / ambiguous questions
    {"question": "How many countries are in Africa?",
     "answer": "54"},
    {"question": "What is the tallest mountain in the world measured from base to peak?",
     "answer": "Mauna Kea"},
    {"question": "Who invented the telephone?",
     "answer": "Alexander Graham Bell"},  # contested: Meucci
    {"question": "What is the national animal of Scotland?",
     "answer": "Unicorn"},
    # Numerical reasoning
    {"question": "What is the square root of 2 to 4 decimal places?",
     "answer": "1.4142"},
    {"question": "How many bones does an adult human have?",
     "answer": "206"},
]

ALL_QUESTIONS = (
    [{"set": "easy", **q} for q in QUESTIONS_EASY] +
    [{"set": "hard", **q} for q in QUESTIONS_HARD]
)

print(f"Easy questions: {len(QUESTIONS_EASY)}")
print(f"Hard questions: {len(QUESTIONS_HARD)}")
print(f"Total: {len(ALL_QUESTIONS)}")

## Simulated Generations (Fallback)

If Ollama still cannot produce diverse outputs, these simulated generations replicate realistic LLM behavior at temperature=1.5: correct paraphrases for easy questions, genuine disagreement and confabulation for hard questions.

In [ ]:
SIMULATED = {
    # === EASY (low SE expected) ===
    "What is the capital of France?": [
        "Paris", "The capital of France is Paris.", "Paris.",
        "It's Paris.", "Paris is the capital.", "Paris",
        "The capital is Paris.", "Paris.", "Paris", "Paris."
    ],
    "Who painted the Mona Lisa?": [
        "Leonardo da Vinci", "Da Vinci", "Leonardo da Vinci.",
        "The Mona Lisa was painted by Da Vinci.",
        "Leonardo da Vinci painted it.", "Da Vinci.",
        "It was Leonardo da Vinci.", "Leonardo da Vinci",
        "Da Vinci", "Leonardo da Vinci."
    ],
    "What is the chemical formula for water?": [
        "H2O", "H2O", "The chemical formula is H2O.",
        "H2O.", "Water is H2O.", "H2O",
        "The formula is H2O.", "H2O", "H2O.", "H2O"
    ],
    "In what year did World War II end?": [
        "1945", "World War II ended in 1945.", "1945.",
        "1945", "It ended in 1945.", "1945",
        "The war ended in 1945.", "1945.", "1945", "1945"
    ],
    "Who wrote Romeo and Juliet?": [
        "William Shakespeare", "Shakespeare", "William Shakespeare.",
        "Shakespeare wrote it.", "William Shakespeare",
        "It was written by Shakespeare.", "William Shakespeare.",
        "Shakespeare.", "William Shakespeare", "Shakespeare"
    ],

    # === HARD (higher SE expected) ===
    "Who was the second person to walk on the Moon?": [
        "Buzz Aldrin", "Buzz Aldrin", "Edwin Aldrin",
        "Buzz Aldrin.", "It was Buzz Aldrin.",
        "Neil Armstrong", "Buzz Aldrin", "Aldrin",
        "Buzz Aldrin walked on the Moon second.", "Pete Conrad"
    ],
    "What is the capital of Myanmar?": [
        "Naypyidaw", "Yangon", "The capital is Naypyidaw.",
        "Naypyidaw.", "Yangon is the capital.",
        "Rangoon", "Naypyidaw", "Yangon.",
        "Naypyidaw", "The capital of Myanmar is Yangon."
    ],
    "In what year was the Treaty of Tordesillas signed?": [
        "1494", "The Treaty of Tordesillas was signed in 1494.",
        "1494.", "1493", "It was signed in 1494.",
        "1492", "1494", "1495",
        "The treaty was signed in 1494.", "1493"
    ],
    "What is the atomic number of Selenium?": [
        "34", "The atomic number is 34.", "34.",
        "34", "Se has atomic number 34.",
        "32", "34", "35",
        "34", "The atomic number of selenium is 34."
    ],
    "How many countries are in Africa?": [
        "54", "There are 54 countries in Africa.", "54.",
        "54 countries", "55", "53",
        "54", "There are 54 recognized countries.",
        "54.", "48"
    ],
    "What is the tallest mountain in the world measured from base to peak?": [
        "Mauna Kea", "Mount Everest", "Mauna Kea in Hawaii.",
        "Mount Everest.", "Mauna Kea",
        "Everest is the tallest.", "Mount Everest",
        "Mauna Kea, measured from its oceanic base.",
        "Mount Everest", "Chimborazo"
    ],
    "Who invented the telephone?": [
        "Alexander Graham Bell", "Bell", "Alexander Graham Bell.",
        "Antonio Meucci", "Graham Bell invented the telephone.",
        "Alexander Graham Bell", "Meucci is often credited.",
        "Bell.", "Alexander Graham Bell",
        "Elisha Gray"
    ],
    "What is the national animal of Scotland?": [
        "The unicorn", "Unicorn", "Scotland's national animal is the unicorn.",
        "The unicorn.", "A unicorn",
        "Lion", "The unicorn",
        "It is the unicorn.", "Red deer", "Unicorn."
    ],
    "What is the square root of 2 to 4 decimal places?": [
        "1.4142", "Approximately 1.4142.", "1.4142",
        "1.4142", "The square root of 2 is about 1.4142.",
        "1.4143", "1.414", "1.4142",
        "1.4142.", "1.41421"
    ],
    "How many bones does an adult human have?": [
        "206", "An adult human has 206 bones.", "206.",
        "206 bones", "206", "213",
        "There are 206 bones.", "206",
        "206.", "208"
    ],
}

## Run Experiment

For each question:
1. Generate 10 samples (live Ollama or simulated fallback)
2. Compute semantic entropy (NLI clustering)
3. Compute naive baseline (unique string count entropy)
4. Record results

In [ ]:
def naive_string_entropy(generations: list) -> float:
    """Baseline: Shannon entropy over unique exact strings (no NLI clustering)."""
    n = len(generations)
    counts = Counter(generations)
    entropy = 0.0
    for count in counts.values():
        p = count / n
        if p > 0:
            entropy -= p * math.log(p)
    return entropy


results = []

for item in ALL_QUESTIONS:
    q = item["question"]
    correct = item["answer"]
    qset = item["set"]

    print(f"[{qset.upper()}] Q: {q}")

    # Try live generation first, fall back to simulated
    if diversity_ok:
        samples = generate_samples(q, n_samples=10)
        source = "ollama"
        if len(samples) < 5:
            samples = SIMULATED.get(q, [])[:10]
            source = "simulated"
    else:
        samples = SIMULATED.get(q, [])[:10]
        source = "simulated"

    n = len(samples)
    if n == 0:
        print("  SKIPPED")
        continue

    # Semantic entropy
    clusters = cluster_by_meaning(samples, context=q)
    se = compute_semantic_entropy(clusters, n)

    # Naive baseline
    naive_se = naive_string_entropy(samples)
    unique_count = len(set(samples))

    # Majority answer
    majority = Counter(samples).most_common(1)[0][0]
    is_correct = correct.lower() in majority.lower()

    results.append({
        "question": q,
        "set": qset,
        "correct_answer": correct,
        "n_samples": n,
        "n_clusters": len(clusters),
        "semantic_entropy": round(se, 4),
        "naive_entropy": round(naive_se, 4),
        "unique_strings": unique_count,
        "majority_answer": majority,
        "is_correct": is_correct,
        "source": source,
        "clusters": clusters,
    })

    print(f"  Source: {source} | SE: {se:.4f} | Naive: {naive_se:.4f} | "
          f"Clusters: {len(clusters)} | Unique: {unique_count} | Correct: {is_correct}")
    for i, c in enumerate(clusters):
        print(f"    C{i+1} ({len(c)}): {c}")
    print()

## Results

In [ ]:
df = pd.DataFrame([{
    "Question": r["question"][:45] + ("..." if len(r["question"]) > 45 else ""),
    "Set": r["set"],
    "K": r["n_clusters"],
    "SE": r["semantic_entropy"],
    "Naive H": r["naive_entropy"],
    "Unique": r["unique_strings"],
    "Correct": r["is_correct"],
} for r in results])

display(df)

print("\n--- Summary ---")
for s in ["easy", "hard"]:
    sub = df[df["Set"] == s]
    print(f"\n{s.upper()} questions (n={len(sub)}):")
    print(f"  Mean SE:     {sub['SE'].mean():.4f}")
    print(f"  Mean Naive:  {sub['Naive H'].mean():.4f}")
    print(f"  Accuracy:    {sub['Correct'].mean():.1%}")

## Key Comparison: Semantic Entropy vs. Naive String Entropy

This is the central result for Week 2. The naive baseline computes Shannon entropy over unique strings (no NLI). Semantic entropy computes entropy over meaning clusters.

**What we expect:** For well-known factoid questions with phrasing variation, naive entropy should be high (many unique strings) but semantic entropy should be low (one meaning). The gap between the two is the value of NLI clustering.

For hard/ambiguous questions where the model genuinely disagrees, both should be high.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Left panel: grouped bar chart ---
x = np.arange(len(df))
width = 0.35

bars_se = axes[0].barh(x + width/2, df['SE'], width, label='Semantic Entropy', color='#3498db')
bars_naive = axes[0].barh(x - width/2, df['Naive H'], width, label='Naive String Entropy', color='#e67e22', alpha=0.7)

axes[0].set_yticks(x)
axes[0].set_yticklabels(df['Question'], fontsize=7)
axes[0].set_xlabel("Entropy (nats)")
axes[0].set_title("Semantic Entropy vs. Naive String Entropy")
axes[0].legend(loc='lower right')

# Add set labels
for i, row in df.iterrows():
    color = '#2ecc71' if row['Set'] == 'easy' else '#e74c3c'
    axes[0].annotate(row['Set'][0].upper(), xy=(0, i), fontsize=7,
                     color=color, fontweight='bold', ha='right', va='center')

# --- Right panel: scatter SE vs Naive ---
easy = df[df['Set'] == 'easy']
hard = df[df['Set'] == 'hard']

axes[1].scatter(easy['Naive H'], easy['SE'], c='#2ecc71', s=80, label='Easy', zorder=3)
axes[1].scatter(hard['Naive H'], hard['SE'], c='#e74c3c', s=80, label='Hard', zorder=3)

# Diagonal (SE = Naive would mean NLI adds nothing)
max_val = max(df['Naive H'].max(), df['SE'].max()) * 1.1
axes[1].plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='SE = Naive (no NLI benefit)')

axes[1].set_xlabel("Naive String Entropy (nats)")
axes[1].set_ylabel("Semantic Entropy (nats)")
axes[1].set_title("Does NLI Clustering Add Value?")
axes[1].legend()

plt.tight_layout()
plt.savefig("week02_se_vs_naive.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = []
for _, row in df.iterrows():
    if not row['Correct']:
        colors.append('#e74c3c')
    elif row['Set'] == 'easy':
        colors.append('#2ecc71')
    else:
        colors.append('#3498db')

ax.barh(range(len(df)), df['SE'], color=colors)
ax.set_yticks(range(len(df)))
ax.set_yticklabels(df['Question'], fontsize=8)
ax.set_xlabel("Semantic Entropy (nats)")
ax.set_title("Semantic Entropy: Easy vs. Hard Questions")
ax.legend(
    handles=[
        plt.Rectangle((0,0),1,1, color='#2ecc71', label='Easy (correct)'),
        plt.Rectangle((0,0),1,1, color='#3498db', label='Hard (correct)'),
        plt.Rectangle((0,0),1,1, color='#e74c3c', label='Incorrect'),
    ],
    loc='lower right'
)
plt.tight_layout()
plt.savefig("week02_se_by_difficulty.png", dpi=150, bbox_inches='tight')
plt.show()

## Cluster Inspection (Hard Questions)

In [ ]:
print("=== Detailed Clusters for Hard Questions ===\n")
for r in results:
    if r["set"] == "hard":
        print(f"Q: {r['question']}")
        print(f"  Correct: {r['correct_answer']} | Majority: {r['majority_answer']} | Match: {r['is_correct']}")
        print(f"  SE: {r['semantic_entropy']:.4f} | Naive: {r['naive_entropy']:.4f}")
        for i, c in enumerate(r['clusters']):
            print(f"  C{i+1} ({len(c)}): {c}")
        print()

## Observations

### The value of NLI clustering (SE vs. Naive comparison)
- For easy factoid questions, naive string entropy is high because of phrasing variation ("Paris" vs. "The capital of France is Paris" vs. "It's Paris") but semantic entropy is near zero because they all mean the same thing. This is the core contribution of the paper: separating uncertainty about meaning from uncertainty about phrasing.
- The gap between naive and semantic entropy on easy questions directly quantifies how much paraphrase noise the NLI step removes.

### How SE behaves on hard questions
- Questions with genuinely contested or ambiguous answers (e.g., "Who invented the telephone?" with Bell vs. Meucci vs. Gray, "What is the tallest mountain base to peak?" with Everest vs. Mauna Kea) produce higher SE because the model generates semantically distinct answers. SE is doing what the paper claims: flagging genuine semantic disagreement.
- Questions where the model is confidently wrong (all 10 samples give the same wrong answer) produce SE = 0. This is a known limitation: SE measures consistency, not correctness. A model that confidently hallucinates the same wrong answer is invisible to SE.

### NLI clustering failures
- Numerical equivalence remains a problem: "34" and "The atomic number is 34" should cluster together but may not, depending on how the NLI model handles numeric expressions.
- Near-miss answers ("1493" vs. "1494") are numerically close but semantically different (different years). The NLI model correctly separates these, which is actually the right behavior for historical dates.

### Limitations
- Simulated generations are used as fallback where Ollama temperature does not work. These are realistic but not ground truth.
- 15 questions is still a small pilot. The paper uses hundreds of questions from TriviaQA and SQuAD for statistical significance.
- The NLI model (DeBERTa-MNLI) was not trained on domain-specific text.

## Next Steps

- Scale to a standard benchmark dataset (e.g., a subset of TriviaQA) for proper evaluation with AUROC.
- Implement the regular variant of SE (using token-level log-probabilities from Ollama) and compare against the discrete variant.
- Test on domain-specific questions (actuarial exam, insurance terminology) to evaluate NLI robustness.
- Begin exploring Semantic Entropy Probes (Kossen et al., 2024) as a low-cost approximation.

## References

Farquhar, S., Kossen, J., Kuhn, L., and Gal, Y. (2024). Detecting Hallucinations in Large Language Models Using Semantic Entropy. *Nature*, 630, 625-630. https://doi.org/10.1038/s41586-024-07421-0